<a href="https://colab.research.google.com/github/mundundan-star/online-retail-customer-analytics/blob/main/2.0%20Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
#!pip install boto3 s3fs

#### 1.0 INTRODUCTION
The main purpose of this notebook is to establish a centralised station to conduct feature engineering and manage features that flow into the models created in subsequent notebooks.

In this notebook, functions are created to pull the necessary features from the `fact_online_retail_clean` dataset in AWS S3. This is so that features can be extracted from new raw data that passes through the data cleaning pipeline, by simply calling the function on the data.

The main `feature_engineering` function stored in S3, is designed to pull features for the, segmentation, churn and Revenue prediction notebooks, for model training, validation, and deployment.

In [6]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import userdata

import boto3
import sys
import os
import importlib

bucket_name = 'sales-data-analytics-portfolio-2026'

#Intializing s3 client with credentials
s3 = boto3.client(
    "s3",
    aws_access_key_id = userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key = userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name = 'eu-north-1'
)

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = "eu-north-1"

with open("feature_engineering.py", "w") as f:
    f.write("""
import pandas as pd
import numpy as np
def feature_engineering(df):

    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
    end_date = df["InvoiceDate"].max()

    cust_data = df.groupby("CustomerID")[["InvoiceDate", "StockCode", "Quantity", "UnitPrice", "Revenue"]].agg(
       FirstPurchase = ("InvoiceDate", "min"),
       LastPurchase = ("InvoiceDate", "max"),
       Frequency = ("InvoiceDate", "nunique"),
       ProductDiversity = ("StockCode", "nunique"),
       AvgUnitPrice = ("UnitPrice", "mean"),
       AvgQuantity = ("Quantity", "mean"),
       AOV = ("Revenue", "mean"),
       TotalRevenue = ("Revenue", "sum")
    )

    cust_data["CohortMonth"] = cust_data["FirstPurchase"].dt.to_period("M")
    cust_data["Recency"] = 1 + (end_date - cust_data["LastPurchase"]).dt.days
    cust_data["Tenure"] = 1 + (end_date - cust_data["FirstPurchase"]).dt.days
    cust_data["ObservedLifeSpan"] = 1 + (cust_data["LastPurchase"] - cust_data["FirstPurchase"]).dt.days

    cust_data.drop(["FirstPurchase","LastPurchase"], axis = 1, inplace = True)

    cust_data["RecencyToTenure"] = cust_data["Recency"]/(cust_data["Tenure"])
    cust_data["ActivePurchaseDensity"] = cust_data["Frequency"]/cust_data["ObservedLifeSpan"]
    cust_data["LifetimePurchaseDensity"] = cust_data["Frequency"]/cust_data["Tenure"]

    cust_data["ActiveMRate"] = cust_data["TotalRevenue"]/cust_data["ObservedLifeSpan"]
    cust_data["LifetimeMRate"] = cust_data["TotalRevenue"]/cust_data["Tenure"]

    ipi = (df[["CustomerID", "InvoiceDate"]].sort_values(["CustomerID", "InvoiceDate"], ascending = False))
    ipi = ipi.groupby(["CustomerID", "InvoiceDate"]).size().reset_index()
    ipi["InterPurchaseInterval"] = np.where(ipi["CustomerID"].shift(1) == ipi["CustomerID"], (ipi["InvoiceDate"] - ipi["InvoiceDate"].shift(1)).dt.days + 1, 1)
    ipi = ipi.groupby("CustomerID").agg(
        AvgIPI = ("InterPurchaseInterval", "mean")
    ).reset_index()

    cust_data = cust_data.merge(ipi[["CustomerID", "AvgIPI"]], on = "CustomerID", how = "left")

    cust_data["RelativeSilence"] = cust_data["Recency"]/cust_data["AvgIPI"]

    return cust_data"""
)


# Uploading file to s3 bucket
s3.upload_file('feature_engineering.py', bucket_name, 'functions/feature_engineering.py')
print(f'Successfully uploaded file to {bucket_name} bucket')

s3.download_file(bucket_name, 'functions/feature_engineering.py', 'feature_engineering.py')

sys.path.append(os.getcwd())

import feature_engineering

importlib.reload(feature_engineering)

from feature_engineering import feature_engineering

parquet_url = "s3://sales-data-analytics-portfolio-2026/parquet_files/fact_online_retail_clean.parquet"
df = pd.read_parquet(parquet_url)

from datetime import datetime

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

cutoff_date = '2011-08-31'
cutoff_date = datetime.strptime(cutoff_date, '%Y-%m-%d')

df1 = feature_engineering(df[df["InvoiceDate"].astype('str') <= '2011-08-31']).copy()
parquet_url = "s3://sales-data-analytics-portfolio-2026/parquet_files/features_before_aug31.parquet"
df1.to_parquet(parquet_url)
print("\nSuccessfully uploaded features before August 31st to S3")

print(f'\nPreview of Features applied to data before {cutoff_date}')
print()
print(df1.info())

df1.head()

Successfully uploaded file to sales-data-analytics-portfolio-2026 bucket


/content/feature_engineering.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])



Successfully uploaded features before August 31st to S3

Preview of Features applied to data before 2011-08-31 00:00:00

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3309 entries, 0 to 3308
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype    
---  ------                   --------------  -----    
 0   CustomerID               3309 non-null   float64  
 1   Frequency                3309 non-null   int64    
 2   ProductDiversity         3309 non-null   int64    
 3   AvgUnitPrice             3309 non-null   float64  
 4   AvgQuantity              3309 non-null   float64  
 5   AOV                      3309 non-null   float64  
 6   TotalRevenue             3309 non-null   float64  
 7   CohortMonth              3309 non-null   period[M]
 8   Recency                  3309 non-null   int64    
 9   Tenure                   3309 non-null   int64    
 10  ObservedLifeSpan         3309 non-null   int64    
 11  RecencyToTenure          3309 non-null

,CustomerID,Frequency,ProductDiversity,AvgUnitPrice,AvgQuantity,AOV,TotalRevenue,CohortMonth,Recency,Tenure,ObservedLifeSpan,RecencyToTenure,ActivePurchaseDensity,LifetimePurchaseDensity,ActiveMRate,LifetimeMRate,AvgIPI,RelativeSilence
0,12346.0,1,1,1.040000,74215.000000,77183.600000,77183.60,2011-01,225,225,1,1.000000,1.000000,0.004444,77183.600000,343.038222,1.000000,225.000000
1,12347.0,5,82,2.797661,12.822581,22.506935,2790.86,2010-12,29,267,238,0.108614,0.021008,0.018727,11.726303,10.452659,48.000000,0.604167
2,12348.0,3,21,0.648400,84.640000,46.689600,1167.24,2010-12,148,257,110,0.575875,0.027273,0.011673,10.611273,4.541790,37.333333,3.964286
3,12350.0,1,16,1.581250,12.250000,18.400000,294.40,2011-02,210,210,1,1.000000,1.000000,0.004762,294.400000,1.401905,1.000000,210.000000
4,12352.0,4,24,3.720606,7.545455,19.439697,641.51,2011-02,162,196,35,0.826531,0.114286,0.020408,18.328857,3.273010,9.500000,17.052632


### 2.0 FEATURE DESCRIPTIONS
 -  `CustomerID, float64`: Customer's unique identifier.

 - `Frequency, int64`: Number of days a client made a purchase.   
 - `ProductDiversity, int64`: Number of unique products purchased.
- `AvgUnitPrice, float64`: Average unit price of all purchases.  
- `AvgQuantity, float64`: Average quantity puchased.  
- `AOV, float64`: Average order value.  
- `TotalRevenue, float64`: Sum of revenue for all transactions.  
- `CohortMonth, period[M]`: The month a customer made their first purchase.
- `Recency, int64`: Number of days between last purchase and dataset end date.   
- `Tenure, int64`: Number of days between a customer's first purchase and dataset end date.   
- `ObservedLifeSpan, int64`: Number of days between last and first purchase.    
- `RecencyToTenure, float64`: Recency to tenure ratio, high ratio signals a high likelihood of churn.
- `ActivePurchaseDensity, float64`: Velocity metric, measuring how often a client purchasez relative to their observed lifespan.
- `LifetimePurchaseDensity, float64`: Velocity metric, measuring how often a client purchases relative to their tenure.
- `ActiveMRate, float64`: Financial velocity metric, average financial value per day during observed lifespan.
  
- `LifetimeMRate, float64`: Financial velocity metric, average financial value per day during tenure.
- `AvgIPI, float64`: Average Inter-Purchase Interval; average number of days between daily purchases.
- `RelativeSilence, float64`: Recency divided by average inter-purchase interval, compares time lapsed with customer's expected purchase cycle.